# Freelance Gig Market Intelligence & Job Matcher
**Domain:** Workforce analytics, weekly demand forecasting, content-based job retrieval  
**Source Data:** Upwork job listings extract (Feb–Mar 2024 snapshot)  
**Stack:** Python, pandas, NumPy, scikit-learn, Matplotlib

## Core Analytical Goals
- Audit raw listing fields, missingness, and duplicate URLs prior to modeling.
- Group job titles into role domains and compare hourly vs. fixed-budget offers.
- Spot high-volume and short-window rising domains across observed months.
- Benchmark a `RandomForestRegressor` against a persistence (last-week) baseline on weekly counts.
- Contrast median hourly midpoints across countries meeting a minimum sample size.
- Build a TF-IDF cosine-similarity search helper for role/skill queries.
- Measure remote/work-from-home phrasing in titles and build simple next-period scenarios.

## Data Scope & Caveats
This extract covers 244,828 Upwork listings (`title`, `link`, `published_date`, `is_hourly`, `hourly_low`, `hourly_high`, `budget`, `country`). Because it captures a narrow two-month window from a single freelance marketplace, findings describe patterns within this file rather than economy-wide employment statistics or guaranteed long-term trajectories.


In [ ]:
# Install required packages if needed: pip install pandas numpy matplotlib scikit-learn

from pathlib import Path
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

candidate_paths = [Path("data/job_postings.csv"), Path("../data/job_postings.csv")]
source_csv = next((p for p in candidate_paths if p.exists()), candidate_paths[0])

raw_df = pd.read_csv(source_csv)
print(f"Loaded {len(raw_df):,} records from {source_csv}")
print("Available columns:", list(raw_df.columns))
raw_df.head(3)


## 1. Schema Check, Cleaning & Data Audit

In [ ]:
required_schema = [
    "title", "link", "published_date", "is_hourly",
    "hourly_low", "hourly_high", "budget", "country"
]
absent_cols = [col for col in required_schema if col not in raw_df.columns]
if absent_cols:
    raise KeyError(f"Missing expected input columns: {absent_cols}")

jobs = raw_df.copy()
jobs["title"] = jobs["title"].fillna("").astype(str).str.strip()
jobs["country"] = jobs["country"].fillna("Unknown").astype(str).str.strip()
jobs.loc[jobs["country"] == "", "country"] = "Unknown"

jobs["published_date"] = pd.to_datetime(jobs["published_date"], errors="coerce", utc=True)
for num_col in ["hourly_low", "hourly_high", "budget"]:
    jobs[num_col] = pd.to_numeric(jobs[num_col], errors="coerce")

bool_map = {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}
jobs["is_hourly"] = (
    jobs["is_hourly"].astype(str).str.strip().str.lower().map(bool_map).fillna(False)
)

# Keep only rows that have a non-empty job title
jobs = jobs[jobs["title"].str.len() > 0].copy()

naive_dt = jobs["published_date"].dt.tz_convert(None)
jobs["post_day"] = naive_dt.dt.floor("D")
jobs["year_month"] = jobs["published_date"].dt.strftime("%Y-%m")
jobs["hourly_mid"] = jobs[["hourly_low", "hourly_high"]].mean(axis=1, skipna=True)
jobs["pricing_model"] = np.where(jobs["is_hourly"], "Hourly", "Fixed / Other")
jobs["normalized_title"] = (
    jobs["title"]
    .str.lower()
    .str.replace(r"[^a-z0-9+#. ]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

audit_table = pd.DataFrame({
    "check_item": [
        "valid_title_rows",
        "duplicate_link_rows",
        "unparsed_or_missing_dates",
        "unknown_country_rows",
        "hourly_tagged_rows",
        "non_null_hourly_midpoints",
        "non_null_fixed_budgets"
    ],
    "count": [
        len(jobs),
        int(jobs["link"].duplicated().sum()),
        int(jobs["published_date"].isna().sum()),
        int((jobs["country"] == "Unknown").sum()),
        int(jobs["is_hourly"].sum()),
        int(jobs["hourly_mid"].notna().sum()),
        int(jobs["budget"].notna().sum())
    ]
})
audit_table


In [ ]:
print("Observed timestamp span:", jobs["published_date"].min(), "->", jobs["published_date"].max())
print("Distinct known countries:", jobs.loc[jobs["country"] != "Unknown", "country"].nunique())
print(f"Summary: {len(jobs):,} titled listings retained; {jobs['link'].duplicated().sum():,} duplicate links kept as-is.")


### Audit Notes
- **Duplicate links** are reported without dropping rows so raw listing volume remains transparent.
- **Missing dates** reduce the sample size available for daily, weekly, and monthly trend slices.
- **Unknown countries** are excluded whenever comparing country-level rates.
- **Compensation coverage** varies between hourly midpoints and fixed budgets, so medians and counts should be read together.


## 2. Role Domains, Compensation & Posting Frequency

In [ ]:
role_rules = {
    "Engineering & Dev": [
        r"\bdeveloper\b", r"\bsoftware\b", r"\bprogrammer\b", r"\bweb app\b",
        r"\bfull[- ]?stack\b", r"\bbackend\b", r"\bfrontend\b", r"\bpython\b", r"\bjavascript\b"
    ],
    "Creative & Media": [
        r"\bdesigner\b", r"\bdesign\b", r"\blogo\b", r"\billustration\b",
        r"\bvideo editor\b", r"\banimation\b", r"\bgraphic\b"
    ],
    "Content & Localization": [
        r"\bwriter\b", r"\bwriting\b", r"\bcopywriter\b", r"\bcontent\b",
        r"\btranslation\b", r"\bproofread\b"
    ],
    "Growth & Marketing": [
        r"\bmarketing\b", r"\bseo\b", r"\bsocial media\b", r"\bads\b",
        r"\bsales\b", r"\blead generation\b", r"\bmedia buyer\b"
    ],
    "Data & AI": [
        r"\bdata\b", r"\bmachine learning\b", r"\bartificial intelligence\b",
        r"\bai\b", r"\banalytics\b", r"\bdashboard\b"
    ],
    "Operations & VA": [
        r"\bvirtual assistant\b", r"\bcustomer support\b", r"\badministrative\b",
        r"\bdata entry\b", r"\bassistant\b"
    ],
    "Accounting & Finance": [
        r"\baccountant\b", r"\bbookkeeping\b", r"\bfinance\b", r"\btax\b", r"\bfinancial\b"
    ]
}

compiled_rules = {
    domain: [re.compile(pat, flags=re.IGNORECASE) for pat in pats]
    for domain, pats in role_rules.items()
}

def assign_role_domain(title_text: str) -> str:
    text = str(title_text)
    for domain, patterns in compiled_rules.items():
        if any(p.search(text) for p in patterns):
            return domain
    return "Unassigned / General"

jobs["role_domain"] = jobs["title"].map(assign_role_domain)
jobs["offer_value"] = np.where(jobs["is_hourly"], jobs["hourly_mid"], jobs["budget"])

comp_stats = (
    jobs.groupby(["role_domain", "pricing_model"], dropna=False)
    .agg(
        listings=("title", "size"),
        hourly_median=("hourly_mid", "median"),
        hourly_mean=("hourly_mid", "mean"),
        fixed_median=("budget", "median"),
        fixed_mean=("budget", "mean")
    )
    .reset_index()
    .sort_values("listings", ascending=False)
)
comp_stats


In [ ]:
phrase_terms = [
    "developer", "software", "programmer", "web app", "full stack", "backend",
    "frontend", "python", "javascript", "designer", "design", "logo",
    "illustration", "video editor", "animation", "graphic", "writer", "writing",
    "copywriter", "content", "translation", "proofread", "marketing", "seo",
    "social media", "ads", "sales", "lead generation", "media buyer", "data",
    "machine learning", "artificial intelligence", "ai", "analytics", "dashboard",
    "virtual assistant", "customer support", "administrative", "data entry",
    "assistant", "accountant", "bookkeeping", "finance", "tax", "financial"
]

term_records = []
for term in sorted(set(phrase_terms)):
    matched = jobs[jobs["normalized_title"].str.contains(r"\b" + re.escape(term) + r"\b", regex=True, na=False)]
    term_records.append({
        "term": term,
        "listings": len(matched),
        "median_hourly_mid": matched["hourly_mid"].median(),
        "median_fixed_budget": matched["budget"].median()
    })

term_summary = pd.DataFrame(term_records).sort_values("listings", ascending=False)
term_summary.head(20)


In [ ]:
hourly_chart_df = comp_stats[comp_stats["hourly_median"].notna()].sort_values("hourly_median", ascending=True)
if not hourly_chart_df.empty:
    labels = hourly_chart_df["role_domain"] + " [" + hourly_chart_df["pricing_model"] + "]"
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(labels, hourly_chart_df["hourly_median"], color="#2b6cb0")
    ax.set_xlabel("Median Hourly Midpoint (Raw Source Units)")
    ax.set_title("Median Hourly Midpoint Across Role Domains")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    top_item = hourly_chart_df.iloc[-1]
    low_item = hourly_chart_df.iloc[0]
    print(f"Top median hourly rate: {top_item['role_domain']} ({top_item['pricing_model']}) -> {top_item['hourly_median']:.2f}")
    print(f"Lowest median hourly rate: {low_item['role_domain']} ({low_item['pricing_model']}) -> {low_item['hourly_median']:.2f}")
else:
    print("No hourly midpoints available to plot.")


### Interpreting Domain Rates & Counts
- Role assignment uses priority ordering over regex rules; listings outside those patterns land in `Unassigned / General`.
- Medians are shown rather than arithmetic means so extreme outliers do not skew typical offers.
- Freelance budgets and hourly ranges are not equivalent to salaried full-time compensation.


## 3. Domain Volume & Month-over-Month Shifts

In [ ]:
domain_counts = (
    jobs.groupby("role_domain", as_index=False)
    .size()
    .rename(columns={"size": "listings"})
    .sort_values("listings", ascending=False)
    .reset_index(drop=True)
)
display(domain_counts)

dated_jobs = jobs.dropna(subset=["published_date", "year_month"]).copy()
monthly_counts = (
    dated_jobs.groupby(["year_month", "role_domain"], as_index=False)
    .size()
    .rename(columns={"size": "listings"})
)
all_months = sorted(dated_jobs["year_month"].unique())

if len(all_months) >= 2:
    pivot_m = monthly_counts.pivot(index="role_domain", columns="year_month", values="listings").fillna(0)
    m_prev, m_curr = all_months[-2], all_months[-1]
    shift_df = pivot_m[[m_prev, m_curr]].copy()
    shift_df["delta"] = shift_df[m_curr] - shift_df[m_prev]
    shift_df["pct_delta"] = np.where(
        shift_df[m_prev] > 0,
        100.0 * shift_df["delta"] / shift_df[m_prev],
        np.nan
    )
    shift_df = shift_df.sort_values("delta", ascending=False).reset_index()
    display(shift_df)
else:
    print("Fewer than two distinct months available for period-over-period comparison.")


In [ ]:
if not monthly_counts.empty:
    top_8_domains = domain_counts.head(8)["role_domain"].tolist()
    trend_slice = monthly_counts[monthly_counts["role_domain"].isin(top_8_domains)]
    fig, ax = plt.subplots(figsize=(10, 5))
    for dom, grp in trend_slice.groupby("role_domain"):
        ax.plot(grp["year_month"], grp["listings"], marker="s", linewidth=2, label=dom)
    ax.set_xlabel("Month")
    ax.set_ylabel("Listings")
    ax.set_title("Monthly Listing Count for Top Role Domains")
    ax.tick_params(axis="x", rotation=40)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## 4. Weekly Demand Forecasting (Random Forest vs. Naive Baseline)

In [ ]:
wk_jobs = dated_jobs.copy()
wk_jobs["week_start"] = wk_jobs["published_date"].dt.tz_convert(None).dt.to_period("W").dt.start_time
wk_counts = (
    wk_jobs.groupby(["week_start", "role_domain"], as_index=False)
    .size()
    .rename(columns={"size": "listings"})
)
week_grid = sorted(wk_counts["week_start"].unique())

eval_rows = []
next_wk_rows = []

if len(week_grid) >= 10:
    for dom in domain_counts.head(5)["role_domain"]:
        s = (
            wk_counts[wk_counts["role_domain"] == dom]
            .set_index("week_start")["listings"]
            .reindex(week_grid, fill_value=0)
            .astype(float)
        )
        feat = pd.DataFrame({"target": s})
        feat["lag1"] = feat["target"].shift(1)
        feat["lag2"] = feat["target"].shift(2)
        feat["lag4"] = feat["target"].shift(4)
        feat["roll4_mean"] = feat["target"].shift(1).rolling(4).mean()
        feat["t_idx"] = np.arange(len(feat), dtype=float)
        feat = feat.dropna()

        if len(feat) < 8:
            continue

        cut = max(5, int(len(feat) * 0.8))
        tr, te = feat.iloc[:cut], feat.iloc[cut:]
        feature_cols = ["lag1", "lag2", "lag4", "roll4_mean", "t_idx"]

        rf = RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=101)
        rf.fit(tr[feature_cols], tr["target"])
        rf_pred = np.clip(rf.predict(te[feature_cols]), 0.0, None)
        naive_pred = te["lag1"].to_numpy()

        eval_rows.append({
            "role_domain": dom,
            "eval_weeks": len(te),
            "rf_mae": mean_absolute_error(te["target"], rf_pred),
            "rf_rmse": float(np.sqrt(mean_squared_error(te["target"], rf_pred))),
            "naive_mae": mean_absolute_error(te["target"], naive_pred),
            "naive_rmse": float(np.sqrt(mean_squared_error(te["target"], naive_pred)))
        })

        full_rf = RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=101)
        full_rf.fit(feat[feature_cols], feat["target"])

        upcoming_X = pd.DataFrame([{
            "lag1": float(feat["target"].iloc[-1]),
            "lag2": float(feat["target"].iloc[-2]),
            "lag4": float(feat["target"].iloc[-4]),
            "roll4_mean": float(feat["target"].iloc[-4:].mean()),
            "t_idx": float(len(feat))
        }])
        next_wk_rows.append({
            "role_domain": dom,
            "last_week_actual": float(feat["target"].iloc[-1]),
            "next_week_rf_est": float(max(0.0, full_rf.predict(upcoming_X)[0]))
        })

    display(pd.DataFrame(eval_rows))
    display(pd.DataFrame(next_wk_rows).sort_values("next_week_rf_est", ascending=False))
else:
    print(f"Only {len(week_grid)} weekly buckets found; need at least 10 for lagged regression.")


### Forecast Evaluation Notes
Models are trained on earlier weeks and tested on later weeks (`80/20` chronological split) to avoid data leakage. Comparing `rf_mae` / `rf_rmse` against `naive_mae` / `naive_rmse` shows whether lag and rolling features beat simply repeating the prior week's volume.


## 5. Country-Level Hourly Rate Comparison

In [ ]:
hourly_valid = jobs[
    jobs["is_hourly"] & jobs["hourly_mid"].notna() & (jobs["country"] != "Unknown")
]

country_bench = (
    hourly_valid.groupby("country", as_index=False)
    .agg(
        hourly_count=("hourly_mid", "size"),
        mean_hourly=("hourly_mid", "mean"),
        median_hourly=("hourly_mid", "median")
    )
    .query("hourly_count >= 20")
    .sort_values("median_hourly", ascending=False)
    .reset_index(drop=True)
)
country_bench.head(25)


In [ ]:
if not country_bench.empty:
    top_countries_plot = country_bench.sort_values("median_hourly", ascending=True).tail(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_countries_plot["country"], top_countries_plot["median_hourly"], color="#2f855a")
    ax.set_xlabel("Median Hourly Midpoint (Unnormalized Source Units)")
    ax.set_title("Top Countries by Median Hourly Midpoint (Min 20 Hourly Listings)")
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    high_c = country_bench.iloc[0]
    low_c = country_bench.iloc[-1]
    print(f"Eligible countries (n >= 20): {len(country_bench)}")
    print(f"Highest median: {high_c['country']} ({high_c['median_hourly']:.2f}, n={int(high_c['hourly_count'])})")
    print(f"Lowest median among eligible: {low_c['country']} ({low_c['median_hourly']:.2f}, n={int(low_c['hourly_count'])})")
else:
    print("Insufficient country records meeting the n >= 20 threshold.")


## 6. TF-IDF Content-Based Job Search Prototype

In [ ]:
catalog = jobs[[
    "title", "link", "country", "role_domain",
    "is_hourly", "hourly_mid", "budget", "published_date"
]].copy()
catalog["search_text"] = catalog["title"].fillna("").str.lower()

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
    max_features=50000
)
title_vectors = tfidf.fit_transform(catalog["search_text"])

def search_matching_postings(query: str, country_filter: str | None = None, top_k: int = 10) -> pd.DataFrame:
    clean_q = str(query).strip()
    if not clean_q:
        raise ValueError("Query string cannot be empty.")
    q_matrix = tfidf.transform([clean_q.lower()])
    sim_scores = cosine_similarity(q_matrix, title_vectors).ravel()

    ranked = catalog.assign(match_score=sim_scores)
    if country_filter:
        ranked = ranked[ranked["country"].str.casefold() == country_filter.strip().casefold()]

    ranked = (
        ranked[ranked["match_score"] > 0]
        .sort_values(["match_score", "published_date"], ascending=[False, False])
        .head(top_k)
    )
    return ranked[[
        "title", "country", "role_domain", "is_hourly",
        "hourly_mid", "budget", "published_date", "link", "match_score"
    ]]

search_matching_postings("Python developer data analysis", top_k=10)


### Search Prototype Caveats
Similarity scores measure lexical overlap between the query and job titles using TF-IDF and cosine similarity; they do not reflect candidate fit, hiring likelihood, or full job descriptions.


## 7. Monthly Activity, Remote Title Signals & Scenario Baselines

In [ ]:
monthly_pulse = (
    dated_jobs.groupby("year_month", as_index=False)
    .agg(
        total_listings=("title", "size"),
        hourly_listings=("is_hourly", "sum"),
        active_countries=("country", "nunique")
    )
)
display(monthly_pulse)

if not monthly_pulse.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(monthly_pulse["year_month"], monthly_pulse["total_listings"], marker="o", color="#805ad5")
    ax.set_xlabel("Month")
    ax.set_ylabel("Listings")
    ax.set_title("Monthly Listing Volume Across Dataset")
    ax.tick_params(axis="x", rotation=40)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    hi_m = monthly_pulse.loc[monthly_pulse["total_listings"].idxmax()]
    lo_m = monthly_pulse.loc[monthly_pulse["total_listings"].idxmin()]
    print(f"Peak month: {hi_m['year_month']} ({int(hi_m['total_listings']):,} listings)")
    print(f"Lowest month: {lo_m['year_month']} ({int(lo_m['total_listings']):,} listings)")


In [ ]:
remote_regex = r"\b(remote|work from home|wfh|anywhere|distributed|virtual)\b"
jobs["has_remote_tag"] = jobs["normalized_title"].str.contains(remote_regex, regex=True, na=False)

remote_overview = pd.DataFrame({
    "metric": ["remote_keyword_titles", "total_titles", "remote_keyword_share_pct"],
    "value": [
        int(jobs["has_remote_tag"].sum()),
        len(jobs),
        round(100.0 * float(jobs["has_remote_tag"].mean()), 2)
    ]
})
display(remote_overview)

remote_by_domain = (
    jobs.groupby("role_domain", as_index=False)["has_remote_tag"]
    .agg(remote_count="sum", total_count="count", share="mean")
)
remote_by_domain["share_pct"] = 100.0 * remote_by_domain["share"]
remote_by_domain = remote_by_domain.drop(columns="share").sort_values("remote_count", ascending=False)
display(remote_by_domain)

if len(all_months) >= 2:
    m_grid = monthly_counts.pivot(index="role_domain", columns="year_month", values="listings").fillna(0)
    m_prev, m_last = all_months[-2], all_months[-1]
    scenario_list = []
    for dom in domain_counts.head(10)["role_domain"]:
        v_prev = float(m_grid.loc[dom, m_prev]) if dom in m_grid.index else 0.0
        v_last = float(m_grid.loc[dom, m_last]) if dom in m_grid.index else 0.0
        step = v_last - v_prev
        scenario_list.append({
            "role_domain": dom,
            "prev_month": v_prev,
            "latest_month": v_last,
            "mom_change": step,
            "flat_next_month": v_last,
            "linear_step_next_month": max(0.0, v_last + step)
        })
    display(pd.DataFrame(scenario_list).sort_values("latest_month", ascending=False))


## 8. Export Processed Tables & Wrap-Up

In [ ]:
export_dir = Path("outputs")
export_dir.mkdir(exist_ok=True)

audit_table.to_csv(export_dir / "audit_summary.csv", index=False)
domain_counts.to_csv(export_dir / "domain_counts.csv", index=False)
comp_stats.to_csv(export_dir / "compensation_by_domain.csv", index=False)
country_bench.to_csv(export_dir / "country_hourly_benchmarks.csv", index=False)
monthly_pulse.to_csv(export_dir / "monthly_pulse.csv", index=False)
remote_by_domain.to_csv(export_dir / "remote_tags_by_domain.csv", index=False)

print("Exported CSV files to:", export_dir.resolve())


### Wrap-Up
This notebook audits the raw Upwork extract, summarizes compensation and domain dynamics, benchmarks a weekly random-forest forecast against a naive baseline, provides a TF-IDF title search prototype, and exports summary tables to `outputs/`. Connecting an authorized live job feed and scheduling automated runs would be required for ongoing production monitoring.
